In [1]:
import sys
import pandas as pd
import numpy as np
import itertools as itt
import matplotlib.pyplot as plt
import seaborn as sns
import petab
import pypesto
from pypesto.store import OptimizationResultHDF5Reader
from pypesto import OptimizeResult, Result
from pypesto.C import MODE_FUN
from pypesto.visualize import waterfall
from petab import get_simulation_conditions

from mEncoder.autoencoder import MechanisticAutoEncoder
from process_data import training_samples, test_samples, Wildcards
from mEncoder.training import create_pypesto_problem
from mEncoder.petab_subproblem import load_petab
from mEncoder import (
    results_dir,
    data_dir,
    fig_dir,
    apply_objective_settings,
    COLLECTED_ESTIMATION_OUTFILE_TEMP,
)
from mEncoder.analysis import process_simulation
from mEncoder.plotting import plot_cross_samples
from training_configuration import L1_INFLATE_REGS, HIDDEN_LAYERS

from pathlib import Path

%matplotlib inline

In [9]:
MODEL = "EGFR_MAPK"
DATA = "dream_cytof"
SAMPLES = "0_5"

result_path = results_dir / MODEL / DATA

In [3]:
samples_train = training_samples(Wildcards(DATA, SAMPLES))
samples_test = test_samples(Wildcards(DATA, SAMPLES))


def load_mae(dataset, hidden_layers, alpha):
    datafiles = (
        data_dir / f"{DATA}__{MODEL}__measurements.tsv",
        data_dir / f"{DATA}__{MODEL}__conditions.tsv",
        data_dir / f"{DATA}__{MODEL}__observables.tsv",
    )

    mae_train = MechanisticAutoEncoder(
        hidden_layers,
        datafiles,
        pathway_name=MODEL,
        samples=samples_train,
        par_modulation_scale=0.0,
    )

    if dataset == "train":
        return mae_train

    return MechanisticAutoEncoder(
        hidden_layers,
        datafiles,
        pathway_name=MODEL,
        samples=samples_test,
        par_modulation_scale=0.0,
        features=mae_train.features,
        imputer=mae_train.imputer,
        scaler=mae_train.scaler,
        pca=mae_train.pca,
    )

In [10]:
def compare_multistart():
    results = []
    legends = []
    for alpha, hidden_layers in itt.product(L1_INFLATE_REGS, HIDDEN_LAYERS):
        mae = load_mae("train", hidden_layers, alpha)
        problem = create_pypesto_problem(mae)
        apply_objective_settings(problem, MODEL)

        infile = result_path / COLLECTED_ESTIMATION_OUTFILE_TEMP.format(
            samples=SAMPLES, n_hidden=hidden_layers, alpha=alpha
        )

        reader = OptimizationResultHDF5Reader(infile)
        result = pypesto.Result(problem)
        result.optimize_result = reader.read().optimize_result
        results.append(result)
        legends.append(f"{alpha} ({hidden_layers})")

    waterfall(results=results, legends=legends)

In [11]:
compare_multistart()

In [4]:
def evaluate_training(dataset):
    evaluations = []
    for alpha, hidden_layers in itt.product(L1_INFLATE_REGS, HIDDEN_LAYERS):
        mae = load_mae(dataset, hidden_layers, alpha)
        problem = create_pypesto_problem(mae)
        apply_objective_settings(problem, MODEL)

        infile = result_path / COLLECTED_ESTIMATION_OUTFILE_TEMP.format(
            samples=SAMPLES, n_hidden=hidden_layers, alpha=alpha
        )

        reader = OptimizationResultHDF5Reader(infile)
        result = pypesto.Result(problem)
        result.optimize_result = reader.read().optimize_result

        x = problem.get_reduced_vector(
            result.optimize_result.list[0]["x"], problem.x_free_indices
        )

        conditions = get_simulation_conditions(
            mae.petab_importer.petab_problem.measurement_df
        )

        res = problem.objective(x, mode=MODE_FUN, return_dict=True)

        if dataset == "train":
            samples = samples_train
        else:
            samples = samples_test

        for sample in samples:
            process_simulation(
                evaluations,
                res,
                conditions,
                sample,
                "full",
                alpha,
                hidden_layers,
            )

    return pd.DataFrame(evaluations)

In [5]:
outdir = fig_dir / MODEL / DATA


def analyse(dataset):
    df = evaluate_training(dataset)
    df.to_csv(outdir / f"{SAMPLES}_evaluate_training_{dataset}.csv")
    g = sns.FacetGrid(
        data=df[df["sample"].apply(lambda x: x.endswith("_dyn"))],
        col="sample",
        hue="layers",
        palette="Blues",
        col_wrap=5,
    )
    g.map_dataframe(sns.lineplot, x="alpha", y="chi2")
    [ax.set(yscale="log") for ax in g.axes]
    plt.tight_layout()
    plt.savefig(outdir / f"{SAMPLES}_evaluate_training_{dataset}.pdf")

In [6]:
analyse("train")

In [7]:
analyse("test")

In [13]:
def evaluate_average(dataset):
    df_meas = pd.read_csv(
        data_dir / f"{DATA}__{MODEL}__measurements.tsv",
        sep="\t",
        index_col=[0],
    )
    df_obs = pd.read_csv(
        data_dir / f"{DATA}__{MODEL}__observables.tsv", sep="\t", index_col=[0]
    )

    df_meas = df_meas[
        df_meas[petab.OBSERVABLE_ID].apply(lambda x: x in df_obs.index)
    ]

    df_train = df_meas[
        df_meas[petab.PREEQUILIBRATION_CONDITION_ID].apply(
            lambda x: x in samples_train
        )
    ]

    df_train["condition"] = df_train[petab.SIMULATION_CONDITION_ID].apply(
        lambda x: x.split("__")[1]
    )

    avg_model = df_train.groupby(
        [petab.OBSERVABLE_ID, petab.TIME, "condition"]
    ).agg(np.nanmean)

    df_sim = df_meas.copy()
    for ir, r in df_meas.iterrows():
        df_sim.loc[ir, petab.MEASUREMENT] = avg_model.loc[
            (
                r.observableId,
                r.time,
                r[petab.SIMULATION_CONDITION_ID].split("__")[1],
            ),
            petab.MEASUREMENT,
        ]

    df_sim[petab.SIMULATION] = df_sim[petab.MEASUREMENT]

    plot_cross_samples(df_meas, df_sim, fig_dir / "average", "avg")

    if dataset == "train":
        samples = samples_train
    else:
        samples = samples_test

    evaluations = []

    for sample in samples:
        s = df_meas[df_meas[petab.PREEQUILIBRATION_CONDITION_ID] == sample]

        chi2 = 0
        for (observable, time, cond), m in avg_model.iterrows():
            d = s[
                (s[petab.OBSERVABLE_ID] == observable)
                & (s[petab.TIME] == time)
                & (s[petab.SIMULATION_CONDITION_ID] == f"{sample}__{cond}")
            ]
            r = (d[petab.MEASUREMENT] - m[petab.MEASUREMENT]) / d[
                petab.NOISE_PARAMETERS
            ]
            chi2 += np.nansum(np.power(r, 2))

        evaluations.append(
            {
                "chi2": chi2,
                "sample": f"{sample}_dyn",
                "type": "average",
                "alpha": 0.0,
                "layers": 0,
            }
        )

    return pd.DataFrame(evaluations)

In [14]:
df_train = evaluate_average("train")
df_train

In [19]:
np.mean(np.log10(df_train.chi2))

In [15]:
df_test = evaluate_average("test")
df_test

In [20]:
np.mean(np.log10(df_test.chi2))